# Customer Segmentation for an Online Retailer## RFM-based clustering to replace generic marketing campaigns**IIB415A · Business Intelligence · Universidad del Desarrollo**Facultad de Ingeniería · Prof. Christopher Castro Araya, MSc. · Semester 2026---### Team 4| # | Member | Student ID ||---|--------|-----------|| 1 | Nicolás Bravo | 20431749-6 || 2 | María José López | 21675751-3 || 3 | Matías Mouat | 21610229-0 || 4 | Martín Olivares | 21644638-0 || 5 | Santiago Page | 21612007-8 |*Note: this team has 5 members instead of 4, authorized by the instructor as an exception under the project guidelines.*---### Business question> **How can we segment our customer base using their purchasing behavior (Recency, Frequency and Monetary value) to identify high-value, at-risk and low-engagement groups for targeted marketing?**Our team acts as an external Business Intelligence unit hired by the marketing department of a UK-based online retailer. The company currently sends **the same campaign to every customer**, which is expensive and inefficient: loyal high-spend customers receive discounts they did not need, and customers about to churn receive nothing that would bring them back.### Dataset| | ||---|---|| **Name** | Online Retail || **Source** | UCI Machine Learning Repository || **Link** | https://archive.ics.uci.edu/dataset/352/online+retail || **Content** | Real transactions from a UK online retailer, Dec 2010 – Dec 2011 || **Unit of analysis** | One row = one invoice line; aggregated to one row per customer |### How to run this notebook1. Place `Online Retail.xlsx` in the same folder as this notebook.2. Run **Restart & Run All**. Every figure and number is reproduced with no manual steps.3. The random seed is fixed once at the top (`RANDOM_SEED = 42`) and reused everywhere.4. Two files are written to the folder at the end: `customer_segments.csv` and `dashboard.html`.### Notebook structure| Stage | Content | Section ||-------|---------|---------|| 1 | Frame & KPIs | §1 || 2 | Prepare data | §2 || 3 | Model & evaluate | §3 || 4 | Communicate | §4 || 5 | Ethics & limitations | §5 || — | Exported files | §6 || — | AI use disclosure | §7 |

---# STAGE 1: Frame & KPIs**Business Question:** How can we segment our customer base using their purchasing behavior (Recency, Frequency, and Monetary value) to identify high-value, at-risk, and low-engagement groups for targeted marketing?**Decision-Maker:** Head of Marketing / CRM Manager. They will use these segments to allocate the retention budget and tailor specific email and ad campaigns.**What changes because of our work:** the single generic campaign is replaced by three differentiated treatments — reward, win-back, and low-cost nurture — each assigned to one segment.**Why segmentation and not prediction:** there is no labelled target in this dataset telling us which customer is "valuable" or "about to churn". Any such label would have to be invented by us, and the model would then simply learn our own arbitrary rule. The honest technique for this question is **unsupervised segmentation**: we let the purchasing behaviour itself define the groups.**Success Metric and KPIs:**| KPI | Type | Definition ||-----|------|-----------|| Launch of distinct campaigns per segment | Business (primary) | Each segment receives a different treatment, aiming to increase retention of at-risk customers || Silhouette Score > 0.5 | Technical | Measured on a **held-out set of customers** the model never saw during training, not on the training data itself || Conversion rate of segment-specific campaigns | Business | Measured next quarter, against a control group || Marketing cost per customer | Business | Expected to fall as low-engagement customers move to a cheaper channel |

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as sns# Set random seed for reproducibility (Rubric requirement)RANDOM_SEED = 42np.random.seed(RANDOM_SEED)# Load the dataset using a relative pathdf = pd.read_excel('Online Retail.xlsx')# Check that the data loaded correctlyprint(f"Rows: {df.shape[0]:,}   Columns: {df.shape[1]}")df.head()

---# STAGE 2: Prepare data (EDA, Cleaning & Features)To build a reliable RFM (Recency, Frequency, Monetary) segmentation model, we need to process the raw transactional data into customer-level metrics.### Data dictionary| Column | Type | Meaning ||--------|------|---------|| `InvoiceNo` | categorical | Invoice number. A code starting with **C** marks a **cancellation** || `StockCode` | categorical | Product code || `Description` | text | Product name || `Quantity` | numeric | Units per line. Negative on cancellations and returns || `InvoiceDate` | datetime | Timestamp of the transaction || `UnitPrice` | numeric | Price per unit, in GBP || `CustomerID` | categorical | Customer identifier. **Missing on a large share of rows** || `Country` | categorical | Country of the customer |## 2.1 Exploratory analysis: what is wrong with this dataBefore cleaning anything we measure the problems. Every cleaning step in §2.2 is justified by a number produced here.

In [ ]:
# ==============================================================================# 1. MISSING VALUES# ==============================================================================missing = pd.DataFrame({    'n_missing': df.isna().sum(),    'pct_missing': (df.isna().mean() * 100).round(2),})missing = missing[missing['n_missing'] > 0].sort_values('n_missing', ascending=False)print('MISSING VALUES')print(missing.to_string() if len(missing) else '  none')# ==============================================================================# 2. QUALITY ISSUES, QUANTIFIED# ==============================================================================n = len(df)issues = pd.DataFrame({    'issue': [        "Cancellation invoices (InvoiceNo starts with 'C')",        'Quantity <= 0 (returns and corrections)',        'UnitPrice <= 0 (free items, admin adjustments)',        'Exact duplicate rows',        'Missing CustomerID',    ],    'rows': [        df['InvoiceNo'].astype(str).str.startswith('C').sum(),        (df['Quantity'] <= 0).sum(),        (df['UnitPrice'] <= 0).sum(),        df.duplicated().sum(),        df['CustomerID'].isna().sum(),    ],})issues['pct_of_total'] = (issues['rows'] / n * 100).round(2)print(f'\nTotal rows: {n:,}\n')print(issues.to_string(index=False))print('\nDATE RANGE')print(f"  from {df['InvoiceDate'].min()}  to  {df['InvoiceDate'].max()}")

In [ ]:
# ==============================================================================# 3. WHERE DO THE CUSTOMERS COME FROM?# ==============================================================================top_countries = df['Country'].value_counts().head(10)fig, ax = plt.subplots(figsize=(9, 4))ax.barh(top_countries.index[::-1], top_countries.values[::-1], color='#0B6E6E')ax.set_xlabel('Number of transaction lines')ax.set_title('Top 10 countries by transaction volume')ax.grid(axis='x', alpha=0.25)plt.tight_layout()plt.show()uk_share = (df['Country'] == 'United Kingdom').mean() * 100print(f'United Kingdom accounts for {uk_share:.1f}% of all transaction lines.')print(f"Distinct countries: {df['Country'].nunique()}")print('\nThis geographic concentration is discussed as a bias in STAGE 5.')

### What the EDA tells us1. **`CustomerID` is missing on a large block of rows.** These are unregistered sales. They cannot be attributed to a customer, and our unit of analysis *is* the customer, so they are unusable here.2. **Cancellations exist and carry negative quantities.** Left in, they would deflate a customer's monetary value.3. **`UnitPrice` includes zeros.** These are administrative adjustments, not sales.4. **Exact duplicate rows exist**, probably from the extraction process.5. **The data is dominated by the United Kingdom** and covers roughly one year.**Why we do not impute the missing `CustomerID`.** Imputation would invent a customer who does not exist, creating a fake purchase history and fabricating a segment. Deletion loses data but keeps the analysis honest; the cost is stated in STAGE 5.## 2.2 Cleaning and feature engineering**Data Cleaning Steps:*** **Duplicates:** removed exact duplicate rows, most likely artifacts from the data extraction process.* **Missing Values:** dropped records with missing `CustomerID`, as they cannot be assigned to a specific user for segmentation.* **Outliers & Anomalies:** filtered out negative quantities (returns and cancellations) and zero-pricing errors to avoid distorting the Monetary value.**Feature Engineering:** we created a `TotalAmount` column (`Quantity * UnitPrice`) and aggregated the data per customer to calculate Recency (days since last purchase), Frequency (number of unique invoices) and Monetary (total spend).**On the snapshot date.** Recency must be measured against a fixed reference point. We use **the day after the last transaction in the dataset**. Using `today()` would be wrong: the data ends in 2011, so every customer would look inactive by thousands of days and Recency would carry no information at all.**Why Frequency counts invoices, not lines.** A customer who buys 40 different products in one order made **one** purchase decision, not 40. Counting lines would confuse basket size with loyalty.

In [ ]:
# ==============================================================================# 1. DATA CLEANING (Duplicates, Missing Values & Outliers)# ==============================================================================# Each step logs how many rows it removes, so nothing is dropped silently.df_clean = df.copy()cleaning_log = []def drop_rows(mask, reason):    global df_clean    before = len(df_clean)    df_clean = df_clean.loc[~mask].copy()    cleaning_log.append({'step': reason,                         'rows_removed': before - len(df_clean),                         'rows_left': len(df_clean)})drop_rows(df_clean.duplicated(),                                 '1. Exact duplicate rows')drop_rows(df_clean['CustomerID'].isna(),                         '2. Missing CustomerID')drop_rows(df_clean['InvoiceNo'].astype(str).str.startswith('C'), '3. Cancellation invoices')drop_rows(df_clean['Quantity'] <= 0,                             '4. Quantity <= 0')drop_rows(df_clean['UnitPrice'] <= 0,                            '5. UnitPrice <= 0')log_df = pd.DataFrame(cleaning_log)log_df['pct_of_original'] = (log_df['rows_removed'] / len(df) * 100).round(2)print('CLEANING LOG')print(log_df.to_string(index=False))print(f'\nOriginal rows : {len(df):,}')print(f'Remaining rows: {len(df_clean):,}  ({len(df_clean)/len(df)*100:.1f}% of original)')print(f"Distinct customers: {df_clean['CustomerID'].nunique():,}")# ==============================================================================# 2. FEATURE ENGINEERING# ==============================================================================# Total amount spent on each transaction linedf_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']# Make sure the date column has the correct datetime formatdf_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])# ==============================================================================# 3. BUILDING THE R-F-M TABLE# ==============================================================================# Reference "today": one day after the last transaction observed in the datasnapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)print(f'\nSnapshot date used for Recency: {snapshot_date}')# Group by CustomerID and compute Recency, Frequency and Monetaryrfm = df_clean.groupby('CustomerID').agg({    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency: days since last purchase    'InvoiceNo': 'nunique',                                   # Frequency: number of distinct invoices    'TotalAmount': 'sum'                                      # Monetary: total spend}).reset_index()rfm.rename(columns={    'InvoiceDate': 'Recency',    'InvoiceNo': 'Frequency',    'TotalAmount': 'Monetary'}, inplace=True)print(f'\nCustomers in the RFM table: {len(rfm):,}')display(rfm.head())

In [ ]:
# ==============================================================================# 4. RFM DISTRIBUTIONS# ==============================================================================FEATURES = ['Recency', 'Frequency', 'Monetary']print('RFM SUMMARY')print(rfm[FEATURES].describe().round(2).to_string())print('\nSKEWNESS (0 = symmetric)')print(rfm[FEATURES].skew().round(2).to_string())fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))for ax, col, color in zip(axes, FEATURES, ['#0B6E6E', '#D98C1F', '#3E6B9B']):    ax.hist(rfm[col], bins=50, color=color, edgecolor='white', linewidth=0.4)    ax.set_title(col)    ax.grid(axis='y', alpha=0.25)axes[0].set_ylabel('Customers')plt.tight_layout()plt.show()print('\nFrequency and Monetary are strongly right-skewed: most customers bought a few')print('times for a modest amount, while a small group of wholesale buyers bought')print('constantly for very large amounts. This matters for K-Means, which minimises')print('squared distances and is therefore sensitive to extreme values.')

## 2.3 Handling the skew: the log1p transformation**Decision: we keep the extreme customers, and transform the scale instead of deleting rows.**The wholesale buyers at the top of the Monetary distribution are not data errors — they arethe single most commercially important group in the file. Deleting them would remove exactlythe customers the marketing department most needs to identify.But leaving the raw scale untouched is also wrong. K-Means minimises **squared** Euclideandistance, so on a heavily skewed scale a handful of extreme customers dominate the objectivefunction: the algorithm spends an entire cluster isolating two or three wholesale buyersinstead of describing the bulk of the customer base. A segment containing one customer ismathematically valid and commercially useless — you cannot run a campaign for one person.We therefore apply **`log1p`** to the three features. `log1p(x) = log(1 + x)` compresses thelong right tail while preserving the ordering of customers, and is safe at zero (Recency canbe 0 for someone who bought on the snapshot day, and `log(0)` is undefined). After thetransform, distances between typical customers and extreme customers become comparable, soK-Means describes the whole base rather than chasing a few outliers.**Important:** the transformation is used **only for the clustering**. Every profile, tableand chart reported to the client in STAGE 4 is expressed in the original units — days,orders and GBP — because that is the only version a marketing manager can act on.

In [ ]:
# ==============================================================================# 5. LOG TRANSFORMATION (used for modelling only, not for reporting)# ==============================================================================# Check first that there are no logically impossible recordsimpossible = (rfm['Monetary'] <= 0) | (rfm['Frequency'] <= 0) | (rfm['Recency'] < 0)print(f'Logically impossible RFM records: {impossible.sum()}')rfm = rfm.loc[~impossible].reset_index(drop=True)# log1p compresses the long right tail while keeping the order of customersrfm_log = np.log1p(rfm[FEATURES])print('\nSKEWNESS: BEFORE -> AFTER log1p')skew_comparison = pd.DataFrame({    'before': rfm[FEATURES].skew().round(2),    'after':  rfm_log.skew().round(2),})print(skew_comparison.to_string())fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))for ax, col, color in zip(axes, FEATURES, ['#0B6E6E', '#D98C1F', '#3E6B9B']):    ax.hist(rfm_log[col], bins=50, color=color, edgecolor='white', linewidth=0.4)    ax.set_title(f'log1p({col})')    ax.grid(axis='y', alpha=0.25)axes[0].set_ylabel('Customers')plt.tight_layout()plt.show()

---# STAGE 3: Model & evaluate**Modeling Approach:**We selected **K-Means Clustering** to segment our customers based on their RFM behavior. Since K-Means is a distance-based algorithm, we feed it the **`log1p`-transformed** features from §2.3 and then scale them with `StandardScaler`, so that Recency, Frequency and Monetary contribute equally to the distance calculation. Without the log step, a handful of wholesale buyers would pull entire clusters towards themselves; without scaling, Monetary would dominate purely because its numbers are larger.**Validation strategy (train / test split):**K-Means has no labeled target, so there is nothing to "leak" in the traditional sense — but the model can still overfit to the specific customers it was fitted on: the centroids land exactly where those points happen to be, and the structure falls apart on customers the model never saw. To check this honestly we follow the same discipline as in supervised learning:**split → fit → predict → evaluate**1. **Split** customers into 80% train / 20% test, *before* any scaling.2. **Fit** the `StandardScaler` and `KMeans` **on the training customers only**.3. **Predict** cluster labels for the test customers with the already-fitted objects.4. **Evaluate** the silhouette score on the test set.**This is where leakage would occur if we were careless.** Calling `scaler.fit_transform()` on the *full* RFM table before splitting would mean the mean and standard deviation used to scale the training data already contained information from the test customers, making the test evaluation optimistic. We therefore call `fit_transform` on train and `transform` (never `fit`) on test.**Defensible choice of k:**Our business framing in STAGE 1 calls for three profiles: high-value, at-risk and low-engagement. We check that choice against the data using the elbow method (inertia) and the silhouette score across a range of k values, both computed on the training set only.**Baseline:**A silhouette score alone does not tell us whether K-Means adds value. We compare it against **RFM quintile scoring** — the traditional, non-ML method a marketing team could apply without a data science team — evaluated on the same held-out customers. If K-Means cannot beat it, the honest recommendation would be to use the simpler method.**Evaluation Metric:**The **Silhouette Score** measures how similar a customer is to its own cluster compared to the nearest other cluster, on a scale from -1 to 1. We target > 0.5 as defined in our technical KPI.

In [ ]:
# ==============================================================================# 1. TRAIN / TEST SPLIT (before scaling, so no information leaks)# ==============================================================================from sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScaler# We split the customer INDEX so that the original-unit table (rfm) and the# log-transformed table (rfm_log) stay perfectly aligned.train_idx, test_idx = train_test_split(rfm.index, test_size=0.2, random_state=RANDOM_SEED)rfm_train, rfm_test = rfm.loc[train_idx], rfm.loc[test_idx]# ==============================================================================# 2. SCALING (fit ONLY on train, transform on test)# ==============================================================================# Applied to the log-transformed features from STAGE 2.3scaler = StandardScaler()X_train = scaler.fit_transform(rfm_log.loc[train_idx])   # fit + transform on trainX_test  = scaler.transform(rfm_log.loc[test_idx])        # transform ONLY on test, never fitprint(f'Training customers: {len(rfm_train):,}')print(f'Test customers    : {len(rfm_test):,}')print('\nScaler parameters were learned from the training set only:')print(pd.DataFrame({'mean': scaler.mean_, 'std': scaler.scale_}, index=FEATURES).round(3).to_string())

In [ ]:
# ==============================================================================# 3. CHOOSING k: ELBOW + SILHOUETTE (computed on the training set only)# ==============================================================================from sklearn.cluster import KMeansfrom sklearn.metrics import silhouette_scorek_range = range(2, 9)inertias = []silhouettes = []for k_candidate in k_range:    km = KMeans(n_clusters=k_candidate, random_state=RANDOM_SEED, n_init=10)    labels = km.fit_predict(X_train)    inertias.append(km.inertia_)    silhouettes.append(silhouette_score(X_train, labels))fig, axes = plt.subplots(1, 2, figsize=(12, 4))axes[0].plot(list(k_range), inertias, marker='o', color='#0B6E6E', linewidth=2)axes[0].set_xlabel('Number of clusters (k)')axes[0].set_ylabel('Inertia (within-cluster sum of squares)')axes[0].set_title('Elbow method (train set)')axes[0].grid(alpha=0.25)axes[1].plot(list(k_range), silhouettes, marker='o', color='#D98C1F', linewidth=2)axes[1].axvline(3, color='#A6484A', linestyle='--', label='k=3 (business choice)')axes[1].set_xlabel('Number of clusters (k)')axes[1].set_ylabel('Silhouette score')axes[1].set_title('Silhouette score by k (train set)')axes[1].legend()axes[1].grid(alpha=0.25)plt.tight_layout()plt.show()best_k = list(k_range)[int(np.argmax(silhouettes))]sil_at_3 = silhouettes[list(k_range).index(3)]print(f'k with the highest silhouette (statistics only): {best_k}  (score = {max(silhouettes):.4f})')print(f'k chosen from the business framing (STAGE 1)   : 3  (score = {sil_at_3:.4f})')print(f'Cost of the business choice                    : {max(silhouettes) - sil_at_3:.4f} silhouette points')print('\nWe choose k=3 because it matches the three marketing profiles the department')print('can actually fund and operate. With k=2 the segmentation says nothing more than')print('"good and bad customers", which marketing already knows; with a large k the team')print('would have to design and pay for campaigns it cannot execute.')

In [ ]:
# ==============================================================================# 4. FIT WITH k=3 ON TRAIN, THEN PREDICT AND EVALUATE ON THE HELD-OUT SET# ==============================================================================k = 3kmeans = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)kmeans.fit(X_train)                      # fit ONLY on traintrain_labels = kmeans.labels_test_labels  = kmeans.predict(X_test)    # predict, never fit, on testsil_train = silhouette_score(X_train, train_labels)sil_test  = silhouette_score(X_test, test_labels)evaluation = pd.DataFrame({'Silhouette score': [sil_train, sil_test]},                          index=['Train', 'Test (held-out)'])print('EVALUATION')print(evaluation.round(4).to_string())print(f'\nDrop from train to test: {sil_train - sil_test:+.4f}')print('A small drop means the segment structure generalises to unseen customers.')if sil_test > 0.5:    print('\nTechnical KPI MET on unseen data: the clusters are well defined and distinct.')else:    print('\nTechnical KPI NOT fully met on unseen data. The segments remain business-')    print('actionable and beat the baseline below, but the boundaries between')    print('neighbouring segments are softer than the target. Reported in STAGE 5.')# Are the cluster proportions stable between train and test?all_clusters = range(k)proportions = pd.DataFrame({    'train_%': pd.Series(train_labels).value_counts(normalize=True).reindex(all_clusters, fill_value=0) * 100,    'test_%':  pd.Series(test_labels).value_counts(normalize=True).reindex(all_clusters, fill_value=0) * 100,}).round(1)print('\nCLUSTER PROPORTIONS: TRAIN vs TEST')print(proportions.to_string())

In [ ]:
# ==============================================================================# 5. BASELINE: traditional RFM quintile scoring (no Machine Learning)# ==============================================================================def rfm_quintile_baseline(frame, n_groups=k):    # Replicates what a marketing team would do by hand: rank every customer into    # quintiles on R, F and M, add the three scores, and cut the result into    # n_groups. Used as the reference point for K-Means.    scored = pd.DataFrame(index=frame.index)    # Recency: FEWER days is better, so the ranking is reversed    scored['R'] = pd.qcut(frame['Recency'].rank(method='first', ascending=False), 5, labels=False)    scored['F'] = pd.qcut(frame['Frequency'].rank(method='first'), 5, labels=False)    scored['M'] = pd.qcut(frame['Monetary'].rank(method='first'), 5, labels=False)    total = scored[['R', 'F', 'M']].sum(axis=1)    return pd.qcut(total.rank(method='first'), n_groups, labels=False).valuesbaseline_labels = rfm_quintile_baseline(rfm_test[FEATURES])sil_baseline = silhouette_score(X_test, baseline_labels)comparison = pd.DataFrame({    'Method': ['RFM quintile scoring (traditional baseline)', f'K-Means (k={k})'],    'Silhouette on held-out test': [sil_baseline, sil_test],})print('BASELINE COMPARISON - both evaluated on the same held-out customers')print(comparison.round(4).to_string(index=False))print(f'\nImprovement of K-Means over the traditional baseline: '      f'{sil_test - sil_baseline:+.4f} silhouette points')

In [ ]:
# ==============================================================================# 6. ASSIGN A SEGMENT TO EVERY CUSTOMER# ==============================================================================# The model is NOT re-trained here. We only apply (transform + predict) the# pipeline already fitted on train, so that every customer in the client's# database carries a segment.X_all = scaler.transform(rfm_log)rfm['Cluster'] = kmeans.predict(X_all)# ==============================================================================# 7. NAME THE SEGMENTS AUTOMATICALLY# ==============================================================================# K-Means returns clusters numbered 0, 1, 2. Those numbers are arbitrary and mean# nothing to a marketing manager. Instead of reading the table and assuming by# hand which cluster is which, the name is derived from each cluster's own# averages, so the notebook stays correct even if the cluster order changes.cluster_means = rfm.groupby('Cluster')[FEATURES].mean()# Composite score: high Frequency + high Monetary + LOW Recency = best customercomposite = (    cluster_means['Frequency'].rank()    + cluster_means['Monetary'].rank()    + cluster_means['Recency'].rank(ascending=False))ranked_clusters = composite.sort_values(ascending=False).index.tolist()SEGMENT_NAMES = ['High-Value / Champions', 'At-Risk / Need Attention', 'Low-Engagement / Lost']cluster_to_segment = {cluster: SEGMENT_NAMES[i] for i, cluster in enumerate(ranked_clusters)}rfm['Segment'] = rfm['Cluster'].map(cluster_to_segment)print('Cluster -> Segment (assigned automatically from each cluster own averages):')for cluster_id in sorted(cluster_to_segment):    print(f'  Cluster {cluster_id} -> {cluster_to_segment[cluster_id]}')print(f'\nSegments assigned to all {len(rfm):,} customers.')

---# STAGE 4: CommunicateWe now profile and visualize the three segments. Because they were named automatically in STAGE 3 from their own averages, the labels below stay correct even if the notebook is re-run and the underlying cluster numbers change.

In [ ]:
# ==============================================================================# STAGE 4: Communicate - Visualizing and Profiling the Segments# ==============================================================================segment_order = SEGMENT_NAMES   # Champions -> At-Risk -> Low-Engagementfig, axes = plt.subplots(1, 3, figsize=(18, 5))short_labels = [s.split(' / ')[0] for s in segment_order]sns.boxplot(x='Segment', y='Recency', data=rfm, order=segment_order,            hue='Segment', hue_order=segment_order, palette='viridis',            legend=False, ax=axes[0])axes[0].set_title('Recency by Segment (Lower is better)')sns.boxplot(x='Segment', y='Frequency', data=rfm, order=segment_order,            hue='Segment', hue_order=segment_order, palette='viridis',            legend=False, ax=axes[1])axes[1].set_title('Frequency by Segment (Higher is better)')axes[1].set_yscale('log')   # Log scale so the extreme values stay readablesns.boxplot(x='Segment', y='Monetary', data=rfm, order=segment_order,            hue='Segment', hue_order=segment_order, palette='viridis',            legend=False, ax=axes[2])axes[2].set_title('Monetary by Segment (Higher is better)')axes[2].set_yscale('log')for ax in axes:    ax.set_xticks(range(len(short_labels)))    ax.set_xticklabels(short_labels)    ax.set_xlabel('')plt.tight_layout()plt.show()# Numerical profile, labelled by segment name rather than cluster numberdashboard_summary = rfm.groupby('Segment')[FEATURES].mean().round(2).loc[segment_order]dashboard_summary['Count'] = rfm.groupby('Segment')['CustomerID'].count().loc[segment_order]display(dashboard_summary)

### Business Interpretation & Actionable RecommendationsThe three segments below were assigned automatically in STAGE 3, based on each cluster's own average Recency, Frequency and Monetary values.* **High-Value / Champions:** bought recently, buy often, and spend the most.    * **Recommendation:** Reward them. Early access to new products, exclusive loyalty programs and VIP customer service. **Not discounts** — they already buy at full price, so a discount here is margin given away for nothing.* **At-Risk / Need Attention:** decent frequency and monetary value, but a high Recency (they have not purchased in a while).    * **Recommendation:** Win them back. Targeted re-engagement emails with limited-time codes or personalized recommendations based on past purchases. This is where the retention budget should go: they have already proved they will pay, and recovering one costs a fraction of acquiring a new customer.* **Low-Engagement / Lost:** high recency, very low frequency and low monetary value.    * **Recommendation:** Limit marketing spend. Move them to a generic automated newsletter rather than expensive targeted ads. This is the line item that funds the win-back campaign above.**How we will know it worked (next quarter):** repeat-purchase rate in At-Risk against a held-out control group, revenue per contacted customer by segment, and total marketing cost per customer.

In [ ]:
# ==============================================================================# STAGE 4: Communicate - Self-contained HTML dashboard# ==============================================================================# The dashboard is regenerated from the notebook on every run, so it can never# drift out of sync with the analysis. Images are embedded as base64, which means# the .html file opens in any browser with no server and no external files.import base64import iodef fig_to_base64(fig):    buf = io.BytesIO()    fig.savefig(buf, format='png', dpi=110, bbox_inches='tight')    plt.close(fig)    return base64.b64encode(buf.getvalue()).decode('utf-8')PALETTE = ['#0B6E6E', '#D98C1F', '#A6484A']# --- Chart A: share of customers vs share of revenuerevenue_by_segment = rfm.groupby('Segment')['Monetary'].sum().loc[segment_order]pct_customers = (dashboard_summary['Count'] / dashboard_summary['Count'].sum() * 100)pct_revenue   = (revenue_by_segment / revenue_by_segment.sum() * 100)figA, ax = plt.subplots(figsize=(8, 4))xpos = np.arange(len(segment_order)); width = 0.38ax.bar(xpos - width/2, pct_customers.values, width, label='% of customers', color='#3E6B9B')ax.bar(xpos + width/2, pct_revenue.values,   width, label='% of revenue',   color='#D98C1F')ax.set_xticks(xpos)ax.set_xticklabels([s.split(' / ')[0] for s in segment_order])ax.set_ylabel('%')ax.set_title('Customer share vs revenue share')ax.legend(frameon=False)ax.grid(axis='y', alpha=0.25)chartA = fig_to_base64(figA)# --- Chart B: average spend per customerfigB, ax = plt.subplots(figsize=(8, 4))ax.bar([s.split(' / ')[0] for s in segment_order],       dashboard_summary['Monetary'].values, color=PALETTE)ax.set_ylabel('GBP')ax.set_title('Average spend per customer')ax.grid(axis='y', alpha=0.25)for i, v in enumerate(dashboard_summary['Monetary'].values):    ax.text(i, v, f'{v:,.0f}', ha='center', va='bottom', fontsize=9)chartB = fig_to_base64(figB)print('Charts encoded for the dashboard.')

In [ ]:
SEGMENT_ACTIONS = {    'High-Value / Champions':   'Loyalty rewards and early access. No discounts.',    'At-Risk / Need Attention': 'Win-back campaign: personal contact + time-limited offer.',    'Low-Engagement / Lost':    'Low-cost automated newsletter only. Stop paid contact.',}print('Marketing action defined for each segment.')

In [ ]:
# --- Build the segment table rowsrows_html = ''for seg in segment_order:    r = dashboard_summary.loc[seg]    rows_html += (        '<tr>'        f"<td class='seg'>{seg}</td>"        f"<td>{r['Count']:,.0f}</td>"        f"<td>{pct_customers[seg]:.1f}%</td>"        f"<td>{r['Recency']:.0f}</td>"        f"<td>{r['Frequency']:.1f}</td>"        f"<td>&pound;{r['Monetary']:,.0f}</td>"        f"<td>{pct_revenue[seg]:.1f}%</td>"        f"<td class='act'>{SEGMENT_ACTIONS[seg]}</td>"        '</tr>'    )CSS = (    'body{font-family:-apple-system,Segoe UI,Roboto,sans-serif;margin:0;'    'background:#f4f6f7;color:#1b2b2b;}'    'header{background:#0B6E6E;color:#fff;padding:26px 34px;}'    'header h1{margin:0 0 4px;font-size:24px;font-weight:600;}'    'header p{margin:0;opacity:.85;font-size:13px;}'    '.wrap{max-width:1100px;margin:0 auto;padding:26px 20px 50px;}'    '.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(190px,1fr));'    'gap:14px;margin-bottom:26px;}'    '.kpi{background:#fff;border-radius:8px;padding:18px 20px;'    'border-left:4px solid #D98C1F;box-shadow:0 1px 3px rgba(0,0,0,.07);}'    '.kpi .v{font-size:27px;font-weight:700;color:#0B6E6E;}'    '.kpi .l{font-size:11px;text-transform:uppercase;letter-spacing:.7px;'    'color:#67787a;margin-top:5px;}'    '.card{background:#fff;border-radius:8px;padding:20px 24px;margin-bottom:20px;'    'box-shadow:0 1px 3px rgba(0,0,0,.07);}'    '.card h2{margin:0 0 14px;font-size:16px;color:#0B6E6E;'    'border-bottom:2px solid #eef1f1;padding-bottom:8px;}'    'img{max-width:100%;height:auto;display:block;}'    'table{width:100%;border-collapse:collapse;font-size:13px;}'    'th{background:#0B6E6E;color:#fff;padding:9px 10px;text-align:left;font-weight:600;}'    'td{padding:9px 10px;border-bottom:1px solid #eef1f1;}'    'tr:nth-child(even) td{background:#fafbfb;}'    'td.seg{font-weight:700;color:#0B6E6E;}'    'td.act{font-size:12px;color:#40585a;}'    'footer{text-align:center;font-size:11px;color:#8a9a9c;padding:16px;}')html = (    "<!DOCTYPE html><html lang='en'><head><meta charset='utf-8'>"    '<title>Customer Segmentation Dashboard | Team 4</title>'    f'<style>{CSS}</style></head><body>'    '<header><h1>Customer Segmentation Dashboard</h1>'    '<p>Online Retail &middot; RFM K-Means segmentation &middot; Team 4 '    '&middot; IIB415A Business Intelligence</p></header>'    "<div class='wrap'>"    "<div class='kpis'>"    f"<div class='kpi'><div class='v'>{len(rfm):,}</div>"    "<div class='l'>Customers segmented</div></div>"    f"<div class='kpi'><div class='v'>&pound;{rfm['Monetary'].sum()/1e6:.2f}M</div>"    "<div class='l'>Total revenue analysed</div></div>"    f"<div class='kpi'><div class='v'>{pct_revenue[segment_order[0]]:.0f}%</div>"    "<div class='l'>Revenue from Champions</div></div>"    f"<div class='kpi'><div class='v'>{sil_test:.2f}</div>"    "<div class='l'>Silhouette (held-out)</div></div>"    '</div>'    "<div class='card'><h2>Segments, profiles and recommended action</h2><table>"    '<tr><th>Segment</th><th>Customers</th><th>% base</th><th>Recency (days)</th>'    '<th>Orders</th><th>Avg spend</th><th>% revenue</th><th>Recommended action</th></tr>'    f'{rows_html}</table></div>'    "<div class='card'><h2>Customer share vs revenue share</h2>"    f"<img src='data:image/png;base64,{chartA}' alt='Customer share versus revenue share'></div>"    "<div class='card'><h2>Average spend per customer</h2>"    f"<img src='data:image/png;base64,{chartB}' alt='Average spend per customer'></div>"    '</div>'    '<footer>Generated automatically from the project notebook &middot; '    'Universidad del Desarrollo &middot; 2026</footer>'    '</body></html>')with open('dashboard.html', 'w', encoding='utf-8') as f:    f.write(html)print('dashboard.html written successfully.')print('Open it in any browser - it is fully self-contained (images embedded).')

---# STAGE 5: Ethics & LimitsAs a responsible Business Intelligence unit, we must acknowledge the ethical implications and limitations of this model before it is deployed by the Marketing team.**1. Privacy & Data Handling:**While the dataset does not contain explicit names, it tracks granular purchasing behavior and `CustomerID`s. This constitutes pseudo-anonymized data under regulations like GDPR. Pseudonymous is not anonymous: the retailer holds the mapping from `CustomerID` to a real business, so a segment label becomes personal data about an identifiable customer. Marketing must ensure that cross-referencing these IDs with the CRM database is strictly access-controlled and that customers have opted in to receive targeted marketing.**2. Bias & Fairness:*** **Geographic bias.** The dataset is heavily skewed towards the United Kingdom (quantified in §2.1). The segments describe UK purchasing behaviour. Applying them to a new market without retraining would systematically misclassify customers whose delivery times, seasonality and typical order sizes differ.* **Survivorship bias.** We removed a large block of rows with no `CustomerID`. If unregistered purchases are more common among a particular type of buyer, those buyers are simply absent from our segmentation and will never receive a targeted treatment.**3. The self-fulfilling prophecy risk:**If we tell the client to stop contacting "Low-Engagement" customers, those customers receive fewer offers, therefore buy less, therefore look even less engaged at the next refresh, and are cut further. The model would then appear to be correct — but only because acting on it made it correct. **Mitigations we recommend:** keep a control group inside Low-Engagement that continues to receive normal contact; recompute the segmentation every quarter so customers can move between segments; and keep a minimum baseline of contact for every segment.**4. Model Limitations:*** **Mathematical assumption:** K-Means assumes clusters are spherical and of similar size, which may oversimplify complex human behavior. Customers near a boundary are assigned with low confidence.* **Lack of context:** RFM only looks at *when* and *how much* a customer buys, entirely ignoring *what* they buy. Two customers in the High-Value segment might purchase completely different product categories, so the marketing team still needs to tailor the content of the emails, not just the timing.* **Seasonality:** with only twelve months of data, a business that legitimately orders once a year cannot be distinguished from one that has churned.* **Descriptive, not causal:** the segmentation says who is at risk, not why, and does not prove that an intervention will work. The win-back campaign should be run as an A/B test against a control group.

---# 6 · Exported filesThe cell below writes the files referenced by the README and the presentation.

In [ ]:
# Customer-level segmentation output (the deliverable to the client)export = rfm[['CustomerID'] + FEATURES + ['Cluster', 'Segment']]export.to_csv('customer_segments.csv', index=False)print('FILES WRITTEN')print('  - customer_segments.csv')print('  - dashboard.html')print('\nHEADLINE NUMBERS FOR THE SLIDES')print(f'  Customers segmented         : {len(rfm):,}')print(f'  Segments                    : {k}')print(f'  Silhouette (held-out test)  : {sil_test:.4f}')print(f'  Silhouette (quintile base)  : {sil_baseline:.4f}')print(f'  Champions share of revenue  : {pct_revenue[segment_order[0]]:.1f}%')print(f"  Revenue in At-Risk segment  : GBP "      f"{rfm.loc[rfm['Segment'] == segment_order[1], 'Monetary'].sum():,.0f}")

---# 7 · AI use disclosureIn line with §7 of the assignment brief, Team 4 discloses the following.**Tool used:** Claude (Anthropic), accessed through the web interface.**What it was used for:*** Discussing how to structure the notebook across the five required BI stages.* Reviewing the wording of the English markdown narrative, since the team's working language is Spanish.* Discussing the justification for specific technical choices — in particular the train/test discipline for an unsupervised model, and the use of a quintile baseline.**What it was NOT used for:*** The business framing, the choice of dataset, the choice of k and the marketing recommendations are the team's own decisions.**Verification.** Every cell in this notebook was executed and checked by the team. Every number reported in the markdown comes from the output of the cell above it. Each member is able to explain and defend the code in the sections assigned to them, as documented in the contribution table of the `README.md`.---*IIB415A · Business Intelligence · Universidad del Desarrollo · Team 4 · 2026*